In [ ]:
# SISO 5G gNB-UE Simulation using AIRSTRAN D 2200
import sys
import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0, 1"

# Add src directory to Python path
sys.path.append(os.path.abspath('../src'))

# Import or install Sionna
try:
    import sionna.rt
except ImportError as e:
    os.system("pip install sionna-rt")
    import sionna.rt

# Other imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import mitsuba as mi
import warnings

# Suppress warnings
warnings.filterwarnings("ignore", message="invalid value encountered in multiply")
warnings.filterwarnings("ignore", category=UserWarning, module="jupyter_client")

# Import relevant components from Sionna RT
from sionna.rt import load_scene, Transmitter, Receiver, Camera, PathSolver
from sionna.rt import AntennaArray, PlanarArray, SceneObject, ITURadioMaterial
from sionna.rt.antenna_pattern import antenna_pattern_registry

scene_xml_path = "../scene/scenes/Duke/scene.xml"
scene = load_scene(scene_xml_path)

In [ ]:
# ============================================
# SISO Configuration: gNB to UE
# ============================================

scene.frequency = 3.65e9  # 3.7 GHz

# Define UE position (fixed to start)
ue_position = [10.0, 0.0, 0.0]   # UE position (x, y, z in meters)

# ============================================
# Antenna Configuration
# ============================================

# gNB antenna: 3GPP TR 38.901 pattern (AIRSTRAN D 2200)
gnb_pattern_factory = antenna_pattern_registry.get("tr38901")
gnb_pattern = gnb_pattern_factory(polarization="V")

# Friendly jammers use 3GPP TR 38.901 directional pattern so their boresight
# orientation is jointly optimised alongside position and power.
friendly_jammer = antenna_pattern_registry.get("iso")
friendly_pattern = friendly_jammer(polarization="V")

# Rx pattern used to take measurements
ue_pattern_factory = antenna_pattern_registry.get("iso")
ue_pattern = ue_pattern_factory(polarization="V")

# SISO: Single antenna element at origin [0, 0, 0] for both TX and RX
single_element = np.array([[0.0, 0.0, 0.0]])  # Shape: (1, 3)

# Configure antenna arrays
scene.tx_array = AntennaArray(
    antenna_pattern=gnb_pattern,
    normalized_positions=single_element.T  # Shape: (3, 1)
)

jammer_array = AntennaArray(
    antenna_pattern=friendly_pattern,
    normalized_positions=single_element.T
)

scene.rx_array = AntennaArray(
    antenna_pattern=ue_pattern,
    normalized_positions=single_element.T  # Shape: (3, 1)
)

# ============================================
# Add Receiver to Scene
# ============================================

# Create UE receiver
rx = Receiver(name="ue", position=ue_position, display_radius=0.03)
scene.add(rx)

# ============================================
# Configure Propagation Environment
# ============================================

# Disable scattering for basic simulation
for radio_material in scene.radio_materials.values():
    radio_material.scattering_coefficient = 0.4


In [ ]:
from scene_parser import extract_building_info
from tx_placement import TxPlacement
# ============================================
# Place gNB on a Specific Building
# ============================================

#building_info = extract_building_info(scene_xml_path, verbose=True)
# Old: 37
#selected_building_id = 33  # Change this to your desired building number
# TxPlacement will create the transmitter if it doesn't exist and place it on the building
# Correct parameter order: (scene, tx_name, scene_xml_path, building_id, offset)
#tx_placer = TxPlacement(scene, "gnb", scene_xml_path, selected_building_id, offset=30.0)
#tx_placer.set_rooftop_center()
# Get reference to the transmitter (already added to scene by TxPlacement)
#tx = tx_placer.tx
# Convert to flat numpy array instead of nested list
#gnb_position = tx.position.numpy().flatten().tolist()
# Point antenna toward UE
#tx.look_at(ue_position)
#print(f"\nSuccess! gNB placed on building {selected_building_id}")
#print(f"Position: {gnb_position}")

# Adding a transmitter (unconstrained)
gnb_position = [-50.0, 50.0, 25.0]
tx = Transmitter(name="tx", position=gnb_position)
scene.add(tx)

# ============================================
# Compute Propagation Paths
# ============================================

# Instantiate path solver
p_solver = PathSolver()

# Compute propagation paths
paths = p_solver(
    scene=scene,
    max_depth=5,
    los=True,
    specular_reflection=True,
    diffuse_reflection=True,
    refraction=False,
    seed=41
)

# ============================================
# Visualize Scene
# ============================================

# Setup camera
cam = Camera(position=(100.0, 100.0, 50.0))
cam.look_at(gnb_position)

# Preview the scene with propagation paths
#scene.preview(
#    paths=paths,
#    resolution=[1000, 1000],
#    clip_at=200,
#    show_orientations=True
#)

In [ ]:
from boresight_pathsolver import create_zone_mask
import numpy as np

map_config = {
    'center': [0.0, 0.0, 0.0],
    'size': [1400, 1400],
    'cell_size': (0.5, 0.5),
    'ground_height': 0.0,
}

extent = [
    map_config['center'][0] - map_config['size'][0] / 2,
    map_config['center'][0] + map_config['size'][0] / 2,
    map_config['center'][1] - map_config['size'][1] / 2,
    map_config['center'][1] + map_config['size'][1] / 2,
]

zone_params = {
    'center': [0.0, 0.0],
    'width': 250,
    'height': 250,
}

zone_mask, naive_look_at, zone_stats = create_zone_mask(
    map_config=map_config,
    zone_type='box',
    origin_point=gnb_position,
    zone_params=zone_params,
    target_height=0.0,
    scene_xml_path=scene_xml_path,
    exclude_buildings=True,
)
print(f"Zone contains {zone_stats['num_cells']} grid cells")
print(f"Zone coverage: {zone_stats['coverage_fraction']*100:.1f}% of map")
print(f"Naive baseline look-at: {zone_stats['look_at_xyz']}")
print(f"Zone centroid: {zone_stats['centroid_xy']}")

In [ ]:
from tx_placement import TxPlacement

#selected_building_id_2 = 21
#tx_placer2 = TxPlacement(scene, 'gnb2', scene_xml_path,
#                          selected_building_id_2, offset=30.0)
#tx_placer2.set_rooftop_center()
#tx2 = tx_placer2.tx
#tx2.look_at([0.0, 600.0, 0.0])

#gnb2_position = [-80.0, -50.0, 25.0]
#tx2 = Transmitter(name="tx2", position=gnb2_position)
#scene.add(tx2)

gnb3_position = [60.0, -120.0, 25.0]
tx3 = Transmitter(name="tx3", position=gnb3_position)
scene.add(tx3)

In [ ]:
from multi_tx_optimizer import TxConfig

# Build TxConfig list — order must match scene insertion order
tx_configs = [
    TxConfig(
        name='tx',
        on_building=False,
        building_id=1,
        zone_params=zone_params,
    ),
    #TxConfig(
    #    name='tx2',
    #    on_building=False,
    #    building_id=2,
    #    zone_params=zone_params,
    #),
    TxConfig(
        name='tx3',
        on_building=False,
        building_id=3,
        zone_params=zone_params,
    )
]

In [ ]:
from boresight_pathsolver import visualize_multi_tx_strata
import matplotlib.pyplot as plt

def strata_callback(iteration, tx_states, tx_configs, jam_positions=None):
    fig = visualize_multi_tx_strata(
        tx_states, tx_configs, map_config, iteration=iteration,
        jam_positions=jam_positions,
    )
    plt.show()
    plt.close(fig)


In [ ]:
from multi_tx_optimizer import optimize_multi_tx, JammerConfig
from angle_utils import azimuth_elevation_to_yaw_pitch
import mitsuba as mi

jam_configs = [
    JammerConfig(name="jam_1", initial_power_dbm=20.0, initial_position=[-250.0, 0]),
    JammerConfig(name="jam_2", initial_power_dbm=20.0, initial_position=[0, -250.0]),
    JammerConfig(name="jam_3", initial_power_dbm=35.0, initial_position=[250.0, 0]),
    JammerConfig(name="jam_4", initial_power_dbm=20.0, initial_position=[0, 250.0]),
]

# Shared hyperparameters for both runs
_opt_kwargs = dict(
    tx_configs=tx_configs,
    map_config=map_config,
    scene_xml_path=scene_xml_path,
    num_sample_points=750,
    learning_rate=1.5,
    num_iterations=55,
    noise_power=1e-10,
    lds='Halton',
    sampler='rejection',
    sampling_strata='full',
    on_iteration_callback=strata_callback,
    gamma_db=0.0,
    sigma_db=3.0,
    lambda_in=7.0,    # protect inside coverage
    lambda_out=14.0,
    lambda_sharp=3.0,
    lambda_pwr=0.0,
    zone_mask=zone_mask,
    boundary_shell_cells=20,
)

# ── Run 1: BS-only (no jammers) ──────────────────────────────────────────────
print("=" * 60)
print("RUN 1: BS-only optimization")
print("=" * 60)
result_bs, _ = optimize_multi_tx(scene=scene, **_opt_kwargs)

# ── Run 2: Jammer optimization, BS frozen at Run 1's final state ─────────────
# freeze_bs=True: BS is locked so Run 1's optimized boresight is preserved.
# lambda_in=2.0: moderate inside protection so jammers can push above 20 dBm
#   without being fully blocked by the inside penalty.
# lambda_pwr=0.0: no power regularisation — jammers find their natural ceiling.
# lambda_sharp=0.5: relaxed so boundary gradient doesn't pin jammer positions.
# Outside sampling now uses a 1.6x ring (was 3x): focuses the 500 outside
#   sample points on the near-zone band where leakage actually lives and
#   where jammers at 250m can compete with the gNB signal.
print("\n" + "=" * 60)
print("RUN 2: Friendly jammer optimization (BS frozen)")
print("=" * 60)
result_jam, jam_scene = optimize_multi_tx(
    scene=scene,
    jammer_array=jammer_array,
    jam_configs=jam_configs,
    freeze_bs=True,
    **{**_opt_kwargs, 'lambda_in': 2.0, 'lambda_pwr': 0.0, 'lambda_sharp': 0.5},
)


In [ ]:
from multi_tx_optimizer import compare_multi_tx_performance
import matplotlib.pyplot as plt

zone_masks_dict = {'tx': zone_mask, 'tx3': zone_mask}
_cmp_kwargs = dict(
    scene=scene, tx_configs=tx_configs, map_config=map_config,
    zone_masks=zone_masks_dict, noise_power=1e-10, gamma_db=0.0,
)

# ── Evaluate Run 1: BS-only ───────────────────────────────────────────────────
print("=" * 60)
print("EVALUATION: BS-only result")
print("=" * 60)
_, _, stats_bs = compare_multi_tx_performance(
    multi_result=result_bs, jam_scene=None, jammer_configs=None, **_cmp_kwargs
)
plt.show()

# Extract the BS-only SINR map so Run 2's left panel shows the actual BS-only result
_sinr_bs_ref = stats_bs["joint"]["sinr_bs_only_map"]

# ── Evaluate Run 2: BS + Jammers ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("EVALUATION: BS + Jammer result")
print("=" * 60)
_, _, stats_jam = compare_multi_tx_performance(
    multi_result=result_jam, jam_scene=jam_scene, jammer_configs=jam_configs,
    sinr_bs_only_ref=_sinr_bs_ref, **_cmp_kwargs
)
plt.show()

# ── Side-by-side containment summary ─────────────────────────────────────────
print("\n" + "=" * 60)
print("COMPARISON:  BS-only  vs  BS+Jammer")
print(f"{'TX':<8} {'metric':<14} {'BS-only':>10} {'BS+Jam':>10}")
print("-" * 45)
for cfg in tx_configs:
    bs  = stats_bs[cfg.name]['containment']
    jam = stats_jam[cfg.name].get('containment_jam', stats_jam[cfg.name]['containment'])
    print(f"{cfg.name:<8} {'rho_leak':<14} {100*bs['rho_leak']:>9.1f}% {100*jam['rho_leak']:>9.1f}%")
    print(f"{cfg.name:<8} {'rho_hole':<14} {100*bs['rho_hole']:>9.1f}% {100*jam['rho_hole']:>9.1f}%")